# RL / Multi-Agent Query Routing — 5 Agents, Online Learning

5 specialist agents (same models as the HybridLLM notebook) sit behind an **orchestrator agent**.
The orchestrator observes a query (state), picks a specialist to delegate to (action), gets a
reward (simulated satisfaction minus normalized cost), and updates its policy online -- continuously,
no separate train/test phase.

Uses the **exact same 15,000-query pool** as the HybridLLM notebook (same seed, same sampling code)
recombined into one stream, since this design has no train/test split -- it all counts as learning.

Resumable: the loop checkpoints every `CHUNK_SIZE` steps (orchestrator weights + logs), so a session
timeout only costs you that chunk.

In [3]:
# ==============================================================================
# NOTE: Package installation code is commented out below because running 'pip install' 
# inside Jupyter causes permission errors on this shared server environment.
# 
# Install packages directly via the Terminal using your personal environment:
#
# STEP 1: Open Terminal (via SSH or JupyterHub -> File -> New -> Terminal)
# STEP 2: Activate your personal environment:
#         source ~/myenv/bin/activate
#
# STEP 3: Install required packages:
#         pip install transformers accelerate datasets sentencepiece scikit-learn protobuf
#
# STEP 4: Restart your notebook kernel:
#         Click "Kernel" -> "Restart Kernel" in the menu above
# ==============================================================================

# import subprocess, sys
# 
# def pip_install(*pkgs):
#     subprocess.run([sys.executable, "-m", "pip", "install", "--user", "-q", "--upgrade", *pkgs], check=True)
# 
# pip_install("transformers", "accelerate", "datasets", "sentencepiece", "scikit-learn", "protobuf")
# print("installed.")

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print("Is CUDA available?:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0))

Is CUDA available?: True
Device count: 1
Device name: NVIDIA L40S


In [2]:
import os, math, json, random, pickle, shutil, time

# Pin this notebook to GPU 0. (GPU 1 on this shared machine is already heavily used by another
# process -- checked via nvidia-smi -- so both notebooks run on GPU 0, one at a time: finish one,
# Kernel > Shut Down Kernel to free its GPU memory, then run the other.)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    AutoModel, BartTokenizer, BartForConditionalGeneration,
)

SEED = 0
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
torch.set_default_dtype(torch.float32)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, " (physical GPU 0, per CUDA_VISIBLE_DEVICES)")
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | total memory: {gpu_mem_gb:.1f} GB")

# --- Local storage, scoped to one project folder (not system-wide) ---
# Model weights go to the DEFAULT Hugging Face cache (~/.cache/huggingface), on purpose:
# that's shared across both notebooks AND persists across your future months of use --
# nothing here gets re-downloaded next time. Only this project's own checkpoints/results
# live in WORK_DIR, so cleanup later is a single `rm -rf` of one small folder.
WORK_DIR = os.path.expanduser("~/hybridllm_project")
CKPT_DIR = os.path.join(WORK_DIR, "ckpt_shared")   # SAME path in both notebooks -> shared checkpoints
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Work dir:", WORK_DIR)

def save_ckpt(name, obj):
    with open(f"{CKPT_DIR}/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)

def load_ckpt(name):
    path = f"{CKPT_DIR}/{name}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

def load_with_retry(model_class, model_id, **kwargs):
    """Retries with a clean re-download if a cached safetensors file is corrupted."""
    try:
        return model_class.from_pretrained(model_id, **kwargs)
    except Exception as e:
        print(f"  [{model_id}] initial load failed ({type(e).__name__}: {e}).")
        print(f"  [{model_id}] clearing cached snapshot and re-downloading cleanly...")
        cache_root = os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "hub")
        repo_folder = "models--" + model_id.replace("/", "--")
        target = os.path.join(cache_root, repo_folder)
        if os.path.exists(target):
            shutil.rmtree(target)
            print(f"  removed {target}")
        return model_class.from_pretrained(model_id, force_download=True, **kwargs)


Device: cuda  (physical GPU 0, per CUDA_VISIBLE_DEVICES)
GPU: NVIDIA L40S | total memory: 47.7 GB
Work dir: /home/bhavana.ravi/hybridllm_project


## Config

`N_TRAIN`/`N_TEST` are the real target scale (10,000 / 5,000). Run the timing probe (later in this
notebook) BEFORE launching the full label-building loop so you know what you're committing to.
Everything expensive below is checkpointed in small chunks, so an interruption only costs you that
chunk -- just re-run the same cell and it resumes.

**Hardware note:** with 2x L40S 48GB, Llama-2-7b-chat fits comfortably in full fp16 (~14GB) -- no
4-bit quantization needed (that was purely a Kaggle 16GB workaround). All 5 models + BARTScorer
together need roughly 25-28GB, well inside one 48GB card.

In [3]:
N_TRAIN = 10000
N_TEST = 5000
MAX_NEW_TOKENS = 64
CHUNK_SIZE = 100            # queries per checkpoint during label-building / RL loop
ROUTER_EPOCHS = 6
ROUTER_LR = 2e-5
ROUTER_BATCH = 16

MODEL_IDS = [
    ("flan-t5-small",   "google/flan-t5-small",            False, 80e6),
    ("flan-t5-base",    "google/flan-t5-base",             False, 250e6),
    ("flan-t5-large",   "google/flan-t5-large",            False, 780e6),
    ("flan-t5-xl",      "google/flan-t5-xl",               False, 3e9),
    ("llama-2-7b-chat", "NousResearch/Llama-2-7b-chat-hf", True,  7e9),
]
MAX_PARAMS = max(p for _, _, _, p in MODEL_IDS)
LARGEST_MODEL_NAME = max(MODEL_IDS, key=lambda m: m[3])[0]
print("Largest model (baseline / cost=1.0):", LARGEST_MODEL_NAME)


Largest model (baseline / cost=1.0): llama-2-7b-chat


## Data — MixInstruct

Same seed + same sampling code in both notebooks -> identical 15,000 queries in both. HybridLLM
splits them 10k/5k; the RL notebook recombines them into one 15k stream (no train/test split there,
per the design).

In [4]:
cached = load_ckpt("examples")
if cached:
    train_examples, test_examples = cached
    print(f"loaded from checkpoint: train={len(train_examples)}  test={len(test_examples)}")
else:
    print("=== Loading MixInstruct ===")
    ds_train = load_dataset("llm-blender/mix-instruct", split="train")
    ds_test = load_dataset("llm-blender/mix-instruct", split="validation")

    def build_query(ex):
        instr = (ex.get("instruction") or "").strip()
        inp = (ex.get("input") or "").strip()
        return (instr + "\n" + inp).strip() if instr else inp

    def sample_examples(ds, n, seed_offset=0):
        rng = random.Random(SEED + seed_offset)
        idx = rng.sample(range(len(ds)), min(n, len(ds)))
        out = []
        for i in idx:
            ex = ds[i]
            q = build_query(ex)
            ref = (ex.get("output") or "").strip()
            if q and ref:
                out.append({"query": q, "reference": ref})
        return out

    train_examples = sample_examples(ds_train, N_TRAIN, seed_offset=1)
    test_examples = sample_examples(ds_test, N_TEST, seed_offset=2)
    save_ckpt("examples", (train_examples, test_examples))
    print(f"train={len(train_examples)}  test={len(test_examples)}")


=== Loading MixInstruct ===


README.md:   0%|          | 0.00/15.1k [00:00<?, ?B/s]

train_data_prepared.jsonl: reconstructing file:   0%|          |  0.00B / 1.38GB            

train_data_prepared.jsonl: downloading bytes:           |  0.00B            

val_data_prepared.jsonl: reconstructing file:   0%|          |  0.00B / 68.7MB            

...

test_data_prepared.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

train=10000  test=5000


## Load the 5 shared models

Same 5 models in both notebooks -> apples-to-apples comparison. Staying in the FLAN-T5 family for
the small end (same architecture, just scaled) plus Llama-2-7b-chat at the top gives a wide, clean
cost gradient without downloading unrelated large models. All loaded in plain fp16, no quantization
needed on this hardware.

In [5]:
MODEL_REGISTRY = []
for name, model_id, is_causal, params in MODEL_IDS:
    print(f"loading {name} ({model_id}) ...")
    tok = AutoTokenizer.from_pretrained(model_id)
    if is_causal:
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        tok.padding_side = "left"
        model = load_with_retry(
            AutoModelForCausalLM, model_id,
            torch_dtype=torch.float16, device_map="auto",
        )
    else:
        model = load_with_retry(
            AutoModelForSeq2SeqLM, model_id, torch_dtype=torch.float16
        ).to(DEVICE)
    model.eval()
    MODEL_REGISTRY.append({"name": name, "tok": tok, "model": model, "causal": is_causal, "params": params})
    print("  done.")

for m in MODEL_REGISTRY:
    m["cost"] = m["params"] / MAX_PARAMS
    print(f"{m['name']:>16}: params={m['params']:.2e}  normalized cost={m['cost']:.4f}")

N_ARMS = len(MODEL_REGISTRY)
MODEL_NAMES = [m["name"] for m in MODEL_REGISTRY]
LARGEST_IDX = MODEL_NAMES.index(LARGEST_MODEL_NAME)

def generate_one(entry, query, max_new_tokens=MAX_NEW_TOKENS):
    tok, model = entry["tok"], entry["model"]
    enc = tok(query, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        if entry["causal"]:
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
            text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        else:
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
            text = tok.decode(out[0], skip_special_tokens=True)
    return text

print("\nagents ready.")


loading flan-t5-small (google/flan-t5-small) ...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

  done.
   flan-t5-small: params=8.00e+07  normalized cost=0.0114
    flan-t5-base: params=2.50e+08  normalized cost=0.0357
   flan-t5-large: params=7.80e+08  normalized cost=0.1114
      flan-t5-xl: params=3.00e+09  normalized cost=0.4286
 llama-2-7b-chat: params=7.00e+09  normalized cost=1.0000

agents ready.


## BARTScore — quality metric (same as before)

In [7]:
BATCH_SIZE_SCORE = 8
BART_CKPT = "facebook/bart-large-cnn"

class BARTScorer:
    def __init__(self, checkpoint=BART_CKPT, device=DEVICE, max_length=1024):
        self.device = device
        self.max_length = max_length
        self.tokenizer = BartTokenizer.from_pretrained(checkpoint)
        self.model = load_with_retry(BartForConditionalGeneration, checkpoint).to(device)
        self.model.eval()
        self.loss_fct = nn.NLLLoss(reduction="none", ignore_index=self.model.config.pad_token_id)
        self.lsm = nn.LogSoftmax(dim=1)

    @torch.no_grad()
    def score(self, srcs, tgts, batch_size=8):
        scores = []
        for i in range(0, len(srcs), batch_size):
            src_batch = srcs[i:i+batch_size]
            tgt_batch = tgts[i:i+batch_size]
            enc_src = self.tokenizer(src_batch, max_length=self.max_length, truncation=True,
                                      padding=True, return_tensors="pt").to(self.device)
            enc_tgt = self.tokenizer(tgt_batch, max_length=self.max_length, truncation=True,
                                      padding=True, return_tensors="pt").to(self.device)
            tgt_ids = enc_tgt["input_ids"]
            output = self.model(
                input_ids=enc_src["input_ids"], attention_mask=enc_src["attention_mask"],
                labels=tgt_ids,
            )
            logits = output.logits.view(-1, self.model.config.vocab_size)
            loss = self.loss_fct(self.lsm(logits), tgt_ids.view(-1))
            loss = loss.view(tgt_ids.shape[0], -1)
            avg_loss = loss.sum(dim=1) / (tgt_ids != self.model.config.pad_token_id).sum(dim=1)
            scores.extend((-avg_loss).tolist())
        return scores

bart_scorer = BARTScorer()

def quality(hypotheses, references):
    """q(hypothesis) := BARTScore(reference -> hypothesis)."""
    return bart_scorer.score(references, hypotheses, batch_size=BATCH_SIZE_SCORE)

print("BARTScorer ready.")


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BARTScorer ready.


## Combine into one stream

HybridLLM used `train_examples` (10k) for label-building and `test_examples` (5k) for routed
evaluation. The RL design has no such split -- all 15,000 queries are just "learning," so we
concatenate and shuffle them into a single stream.

In [8]:
rl_pool = train_examples + test_examples
random.Random(SEED + 3).shuffle(rl_pool)
print(f"RL pool: {len(rl_pool)} queries (same underlying examples as the HybridLLM notebook)")


RL pool: 15000 queries (same underlying examples as the HybridLLM notebook)


## Reward calibration

Sets the satisfaction threshold τ **without** comparing across models -- the orchestrator only ever
sees the chosen agent's own output, never what the other 4 would have said (that's what keeps this
system's cost structurally lower than HybridLLM's label-building phase). τ is calibrated as the
median BARTScore a middle-of-pack agent (`flan-t5-base`) achieves on a small calibration slice.

**Important caveat, worth stating to your professor explicitly:** in a real deployment this +1/-1
comes from an actual human reacting to the answer. Rating 15,000 generations by hand isn't feasible
here, so this is an automated stand-in -- the same kind of substitution the HybridLLM paper itself
makes (its "quality" signal is also not a live human). The architecture is designed so this is a
one-line swap later: replace the BARTScore-threshold check below with a real user's thumbs up/down,
nothing else in the orchestrator changes.

In [9]:
calib_pool = rl_pool[:40]
calib_queries = [e["query"] for e in calib_pool]
calib_refs = [e["reference"] for e in calib_pool]

calib_agent = next(m for m in MODEL_REGISTRY if m["name"] == "flan-t5-base")
calib_out = [generate_one(calib_agent, q) for q in calib_queries]
calib_q = quality(calib_out, calib_refs)
TAU = float(np.median(calib_q))
agent_name = calib_agent["name"]
print(f"reward threshold tau = {TAU:.4f} (from {agent_name} on {len(calib_pool)} calibration queries)")


reward threshold tau = -4.5488 (from flan-t5-base on 40 calibration queries)


## The orchestrator agent

State = the current query, encoded by a DeBERTa-v3-small backbone (no memory across queries, per
the "state is each query" design). Its head outputs one predicted value (Q-value) per specialist
agent; it delegates to whichever looks best (ε-greedy exploration/exploitation), observes the real
reward, and updates online.

In [10]:
ROUTER_ID = "microsoft/deberta-v3-small"
LAMBDA_COST = 0.5        # weight on the cost penalty -- tune this
EPS_START, EPS_END = 0.3, 0.05

torch.set_default_dtype(torch.float32)

rl_tok = AutoTokenizer.from_pretrained(ROUTER_ID)

class OrchestratorHead(nn.Module):
    """State (query embedding) -> predicted value of delegating to each specialist agent."""
    def __init__(self, hidden_size, n_agents):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hidden_size, 128), nn.ReLU(), nn.Linear(128, n_agents))
    def forward(self, state):
        return self.net(state)

ckpt_state = load_ckpt("rl_training_state")
if ckpt_state is not None:
    rl_backbone = AutoModel.from_pretrained(ROUTER_ID, torch_dtype=torch.float32).to(DEVICE)
    rl_backbone.load_state_dict(ckpt_state["backbone_state"])
    orchestrator_head = OrchestratorHead(rl_backbone.config.hidden_size, N_ARMS).to(DEVICE)
    orchestrator_head.load_state_dict(ckpt_state["head_state"])
    opt = torch.optim.AdamW(list(rl_backbone.parameters()) + list(orchestrator_head.parameters()), lr=2e-5)
    opt.load_state_dict(ckpt_state["opt_state"])
    start_step = ckpt_state["step"]
    action_log = ckpt_state["action_log"]
    cost_log = ckpt_state["cost_log"]
    reward_log = ckpt_state["reward_log"]
    satisfaction_log = ckpt_state["satisfaction_log"]
    print(f"resumed orchestrator from checkpoint at step {start_step}")
else:
    rl_backbone = AutoModel.from_pretrained(ROUTER_ID, torch_dtype=torch.float32).to(DEVICE)
    orchestrator_head = OrchestratorHead(rl_backbone.config.hidden_size, N_ARMS).to(DEVICE)
    opt = torch.optim.AdamW(list(rl_backbone.parameters()) + list(orchestrator_head.parameters()), lr=2e-5)
    start_step = 0
    action_log, cost_log, reward_log, satisfaction_log = [], [], [], []
    print("orchestrator initialized fresh.")


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  286MB            

...

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


orchestrator initialized fresh.


model.safetensors: reconstructing file:   0%|          |  0.00B /  286MB            

model.safetensors: downloading bytes:           |  0.00B            

## Online RL loop

Runs through `rl_pool` once, learning continuously. Checkpointed every `CHUNK_SIZE` steps
(orchestrator weights + optimizer + all logs), so re-running this cell after an interruption
resumes exactly where it left off instead of restarting.

In [11]:
N_STEPS = len(rl_pool)

def save_rl_checkpoint(step):
    save_ckpt("rl_training_state", {
        "step": step,
        "backbone_state": rl_backbone.state_dict(),
        "head_state": orchestrator_head.state_dict(),
        "opt_state": opt.state_dict(),
        "action_log": action_log, "cost_log": cost_log,
        "reward_log": reward_log, "satisfaction_log": satisfaction_log,
    })

t_start = time.time()
for step in range(start_step, N_STEPS):
    ex = rl_pool[step]
    q_text, ref = ex["query"], ex["reference"]
    eps = EPS_START + (EPS_END - EPS_START) * (step / max(1, N_STEPS - 1))

    enc = rl_tok(q_text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    out = rl_backbone(**enc)
    state = out.last_hidden_state[:, 0]
    agent_values = orchestrator_head(state).squeeze(0)

    if np.random.rand() < eps:
        action = np.random.randint(N_ARMS)
    else:
        action = int(torch.argmax(agent_values).item())

    agent = MODEL_REGISTRY[action]
    gen_text = generate_one(agent, q_text)
    q_score = quality([gen_text], [ref])[0]
    satisfaction = 1.0 if q_score >= TAU else -1.0
    reward = satisfaction - LAMBDA_COST * agent["cost"]

    loss = F.mse_loss(agent_values[action], torch.tensor(reward, dtype=torch.float32, device=DEVICE))
    opt.zero_grad(); loss.backward(); opt.step()

    action_log.append(agent["name"]); cost_log.append(agent["cost"])
    reward_log.append(reward); satisfaction_log.append(satisfaction)

    if (step + 1) % CHUNK_SIZE == 0 or step == N_STEPS - 1:
        save_rl_checkpoint(step + 1)
        elapsed = time.time() - t_start
        done_this_session = step + 1 - start_step
        rate = elapsed / max(done_this_session, 1)
        remaining = (N_STEPS - step - 1) * rate
        recent = reward_log[-CHUNK_SIZE:]
        print(f"step {step+1:6d}/{N_STEPS} | eps={eps:.2f} | last={agent['name']:>16} | "
              f"avg reward (last {CHUNK_SIZE})={np.mean(recent):+.3f} | "
              f"est. remaining: {remaining/3600:.2f}h")

print("\nRL loop complete.")
print("Delegation pattern (whole run):", Counter(action_log))
print("Delegation pattern (last 20%):", Counter(action_log[int(0.8 * N_STEPS):]))


step    100/15000 | eps=0.30 | last=   flan-t5-small | avg reward (last 100)=+0.204 | est. remaining: 1.46h
step    200/15000 | eps=0.30 | last=   flan-t5-large | avg reward (last 100)=+0.264 | est. remaining: 1.83h
step    300/15000 | eps=0.30 | last=    flan-t5-base | avg reward (last 100)=+0.176 | est. remaining: 1.90h
step    400/15000 | eps=0.29 | last=      flan-t5-xl | avg reward (last 100)=+0.084 | est. remaining: 1.98h
step    500/15000 | eps=0.29 | last= llama-2-7b-chat | avg reward (last 100)=+0.188 | est. remaining: 2.26h
step    600/15000 | eps=0.29 | last=   flan-t5-large | avg reward (last 100)=+0.302 | est. remaining: 2.53h
step    700/15000 | eps=0.29 | last=      flan-t5-xl | avg reward (last 100)=+0.214 | est. remaining: 2.88h
step    800/15000 | eps=0.29 | last= llama-2-7b-chat | avg reward (last 100)=+0.329 | est. remaining: 3.12h
step    900/15000 | eps=0.29 | last=   flan-t5-large | avg reward (last 100)=+0.212 | est. remaining: 3.24h
step   1000/15000 | eps=0.28

## Cost totals + results export

In [12]:
tail = action_log[int(0.8 * len(action_log)):]

rl_total_norm_cost = float(sum(cost_log))
rl_cost_per_query_norm = rl_total_norm_cost / len(cost_log)

PARAMS = {m["name"]: m["params"] for m in MODEL_REGISTRY}
rl_total_raw_cost = float(sum(PARAMS[name] for name in action_log))

mean_reward = float(np.mean(reward_log))
mean_reward_last20pct = float(np.mean(reward_log[int(0.8 * len(reward_log)):]))
mean_satisfaction = float(np.mean(satisfaction_log))

print(f"pilot size (= full 15,000-query design, no separate train/test): {len(rl_pool)} queries")
print(f"total cost (normalized):      {rl_total_norm_cost:.3f}")
print(f"total cost (raw params):      {rl_total_raw_cost:,.0f}")
print(f"mean reward (whole run):      {mean_reward:+.3f}")
print(f"mean reward (last 20%):       {mean_reward_last20pct:+.3f}")
print(f"mean satisfaction:            {mean_satisfaction:+.3f}")

results = {
    "system": "rl_multiagent_orchestrator",
    "n_queries": len(rl_pool),
    "n_agents": N_ARMS,
    "agents": [{"name": m["name"], "params": m["params"], "cost": m["cost"]} for m in MODEL_REGISTRY],
    "lambda_cost": LAMBDA_COST,
    "tau_satisfaction_threshold": TAU,
    "eps_start": EPS_START, "eps_end": EPS_END,
    "total_cost_normalized": rl_total_norm_cost,
    "total_cost_raw_params": rl_total_raw_cost,
    "cost_per_query_normalized": rl_cost_per_query_norm,
    "mean_reward": mean_reward,
    "mean_reward_last20pct": mean_reward_last20pct,
    "mean_satisfaction": mean_satisfaction,
    "action_distribution_full_run": dict(Counter(action_log)),
    "action_distribution_last20pct": dict(Counter(tail)),
    "action_log": action_log, "cost_log": cost_log, "reward_log": reward_log,
}

with open(os.path.join(OUTPUT_DIR, "rl_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print("saved " + os.path.join(OUTPUT_DIR, "rl_results.json"))


pilot size (= full 15,000-query design, no separate train/test): 15000 queries
total cost (normalized):      8123.207
total cost (raw params):      56,862,450,000,000
mean reward (whole run):      +0.385
mean reward (last 20%):       +0.454
mean satisfaction:            +0.656
saved /home/bhavana.ravi/hybridllm_project/outputs/rl_results.json


## Compare against HybridLLM (run after copying `hybrid_llm_results.json` into this notebook's working dir)

In [14]:
hybrid_path = os.path.join(OUTPUT_DIR, "hybrid_llm_results.json")
pairs_path  = os.path.join(OUTPUT_DIR, "hybridllm_pairs_comparison.json")
rl_total = rl_total_raw_cost

if os.path.exists(hybrid_path):
    with open(hybrid_path) as f:
        hybrid = json.load(f)

    h_label = hybrid["label_building_cost_raw_params"]
    h_best  = hybrid["total_cost_raw_params_best_case"]
    h_mid   = hybrid["total_cost_raw_params"]
    h_worst = hybrid["total_cost_raw_params_worst_case"]

    print(f"HybridLLM ({hybrid['small_model']} + {hybrid['large_model']}), 10k train + 5k test")
    print(f"  - label-building (10k x both models): {h_label:,.0f}")
    print(f"  - total best / mid / worst:           {h_best:,.0f} / {h_mid:,.0f} / {h_worst:,.0f}")
    print(f"RL total (raw params, 15k queries):     {rl_total:,.0f}\n")
    print(f"HybridLLM / RL cost ratio (best / mid / worst): "
          f"{h_best/rl_total:.2f}x / {h_mid/rl_total:.2f}x / {h_worst/rl_total:.2f}x")

    if os.path.exists(pairs_path):
        with open(pairs_path) as f:
            pairs = json.load(f)["pairs"]
        print("\nAll three HybridLLM pairs vs RL:")
        for name, p in pairs.items():
            print(f"  {name:>11}: mid={p['total_cost_raw_params_midpoint']:,.0f}  "
                  f"-> {p['total_cost_raw_params_midpoint']/rl_total:.2f}x RL")
else:
    print("hybrid_llm_results.json not found at " + hybrid_path)

HybridLLM (flan-t5-small + llama-2-7b-chat), 10k train + 5k test
  - label-building (10k x both models): 70,800,000,000,000
  - total best / mid / worst:           71,200,000,000,000 / 88,500,000,000,000 / 105,800,000,000,000
RL total (raw params, 15k queries):     56,862,450,000,000

HybridLLM / RL cost ratio (best / mid / worst): 1.25x / 1.56x / 1.86x

All three HybridLLM pairs vs RL:
    small_gap: mid=125,000,000,000,000  -> 2.20x RL
   medium_gap: mid=97,250,000,000,000  -> 1.71x RL
    large_gap: mid=88,500,000,000,000  -> 1.56x RL
